In [77]:
import h5py
import numpy as np

exp_path = './exp.h5'
exp_reco_path = './exp_reco.h5'

In [ ]:
with h5py.File(exp_path) as he:
    with h5py.File(exp_reco_path) as hr:
        e_group = he['exp']
        r_group = hr['exp']
        
        parts = list(r_group['clusters_centers'].keys())
        part_name = parts[-1]
        print(part_name)
        
        print(he[f'exp/clusters_centers/{part_name}/data'][:])
        print(hr[f'exp/clusters_centers/{part_name}/data'][:])
        
        print(e_group['coords_are_cluster_centered/data'][()])
        print(r_group['coords_are_cluster_centered/data'][()])
        
        print(e_group[f'raw/channels/{part_name}/data'][:])
        print(r_group[f'raw/channels/{part_name}/data'][:])
        
        # First reco event
        r_ev_starts = r_group[f"raw/ev_starts/{part_name}/data"][:]
        r_s, r_e = r_ev_starts[0], r_ev_starts[1]
        first_reco_event_ch = r_group[f'raw/channels/{part_name}/data'][r_s:r_e]
        r_t = r_group[f'raw/data/{part_name}/data'][r_s:r_e, 1]
        r_z = r_group[f'raw/data/{part_name}/data'][r_s:r_e, 4]
        
        
        # Find match
        e_ev_starts = e_group[f"raw/ev_starts/{part_name}/data"][:]
        for e_s, e_e in zip(e_ev_starts[:-1], e_ev_starts[1:]):
            e_channels = e_group[f'raw/channels/{part_name}/data'][e_s:e_e]
            e_t = r_group[f'raw/data/{part_name}/data'][e_s:e_e, 1]
            e_z = r_group[f'raw/data/{part_name}/data'][e_s:e_e, 4]
            if np.array_equal(e_channels, first_reco_event_ch):
                print(f"Found event! Hits num {e_e-e_s}")
                
                
                print(f"Z mean: {r_z.mean()} VS {e_z.mean()}")
                break

part_s2020_c07_r0406
[[-505.73340666 -238.98833701  362.44285031]]
[[-505.67005882 -239.01661486  362.44916682]]
True
True
[149 267 224 ... 159 276 171]
[224 285 198 ...  96 171  20]
Found event! Hits num 110
Z mean: 37.261043548583984 VS 90.46699523925781


In [79]:
first_reco_event_ch

array([224, 285, 198, 286, 171,  68, 132, 248, 178, 133, 113, 258,  97,
        55, 210, 286, 239, 243, 133, 168,  59, 100,  48, 242, 105, 127,
        45, 100, 179, 118, 133, 247, 140,  99, 142, 230, 194, 194, 160,
       194, 286,  84, 224,  64, 132,  20,  57, 284,  19,  21,  22,  17,
        16, 286,  45,  15,  71, 137,  14, 217,  22,  91,  15,  13,  58,
        51, 265,  47, 262,   3, 135, 178, 163,  44, 121, 191, 247, 274,
       277,  76,  74, 212, 253, 109, 168,  39, 280, 267, 230, 271, 246,
       248,  48, 230,  49, 153, 287, 108, 249,  86, 264,  74, 140, 281,
       258, 244, 118, 214,  46, 207], dtype=int32)

In [ ]:
first_reco_event_ch, e_channels

(array([224, 285, 198, 286, 171,  68, 132, 248, 178, 133, 113, 258,  97,
         55, 210, 286, 239, 243, 133, 168,  59, 100,  48, 242, 105, 127,
         45, 100, 179, 118, 133, 247, 140,  99, 142, 230, 194, 194, 160,
        194, 286,  84, 224,  64, 132,  20,  57, 284,  19,  21,  22,  17,
         16, 286,  45,  15,  71, 137,  14, 217,  22,  91,  15,  13,  58,
         51, 265,  47, 262,   3, 135, 178, 163,  44, 121, 191, 247, 274,
        277,  76,  74, 212, 253, 109, 168,  39, 280, 267, 230, 271, 246,
        248,  48, 230,  49, 153, 287, 108, 249,  86, 264,  74, 140, 281,
        258, 244, 118, 214,  46, 207], dtype=int32),
 array([224, 285, 198, 286, 171,  68, 132, 248, 178, 133, 113, 258,  97,
         55, 210, 286, 239, 243, 133, 168,  59, 100,  48, 242, 105, 127,
         45, 100, 179, 118, 133, 247, 140,  99, 142, 230, 194, 194, 160,
        194, 286,  84, 224,  64, 132,  20,  57, 284,  19,  21,  22,  17,
         16, 286,  45,  15,  71, 137,  14, 217,  22,  91,  15,  13,  58

In [83]:
e_z[0], r_z[0]

(np.float32(157.48491), np.float32(-142.39891))

In [ ]:
import uproot as ur

r_rf_path = "/home/albert/Baikal2025/data_manager/exp_reco_root/root_files/2020_cl7_run406_scl_nu_DATA2020.root"
with ur.open(r_rf_path) as rf:
    e_channels = rf['Events/BEvent./BEvent.fPulses/BEvent.fPulses.fChannelID'].array(library='np')[1:]
    active_clusters = [np.unique(ch // 288) for ch in e_channels]
    num_un_clusters = np.array([len(cl) for cl in active_clusters])
    
print(e_channels)

[array([  3,  13,  14,  15,  15,  16,  17,  19,  20,  21,  22,  22,  39,
         44,  45,  45,  46,  47,  48,  48,  49,  51,  55,  57,  58,  59,
         64,  68,  71,  74,  74,  76,  84,  86,  91,  97,  99, 100, 100,
        105, 108, 109, 113, 118, 118, 121, 127, 132, 132, 133, 133, 133,
        135, 137, 140, 140, 142, 153, 160, 163, 168, 168, 171, 178, 178,
        179, 191, 194, 194, 194, 198, 207, 210, 212, 214, 217, 224, 224,
        230, 230, 230, 239, 242, 243, 244, 246, 247, 247, 248, 248, 249,
        253, 258, 258, 262, 264, 265, 267, 271, 274, 277, 280, 281, 284,
        285, 286, 286, 286, 286, 287], dtype=int32)
 array([  0,   4,   8,   9,  10,  20,  22,  23,  36,  36,  36,  36,  40,
         42,  42,  43,  44,  44,  44,  45,  45,  45,  46,  46,  48,  49,
         51,  55,  60,  60,  61,  64,  71,  75,  77,  78,  78,  79,  80,
         81,  82,  84,  85,  87,  87,  97,  98, 100, 100, 103, 105, 119,
        119, 121, 125, 126, 130, 130, 131, 133, 133, 134, 136, 138, 141,

In [116]:
import awkward as ak
e_rf_path = "/net/62/home/albert/Baikal/Data/exp_root_files/s2020_c07_r0406.root"
path_geometry = "Events/BGeomTel./BGeomTel.BGeomTel/BGeomTel.BGeomTel.fOMs/BGeomTel.BGeomTel.fOMs.fPosition"
st = 0

with ur.open(e_rf_path) as rf:
    e_channels = rf['Events/BEvent./BEvent.fPulses/BEvent.fPulses.fChannelID'].array(library='np')[st:]
    coordinates = np.array(ak.unzip(rf[path_geometry].array()))[:,st:]
    active_clusters = [np.unique(ch // 288) for ch in e_channels]
    num_un_clusters = np.array([len(cl) for cl in active_clusters])
    
e_channels

array([array([ 19,  23,  43,  48,  55,  60,  62,  67,  70,  70,  73,  77,  85,
               85,  86,  95, 101, 103, 111, 112, 112, 114, 117, 123, 127, 128,
              128, 128, 128, 139, 142, 145, 145, 145, 145, 145, 147, 149, 157,
              160, 160, 160, 163, 175, 176, 179, 202, 202, 203, 203, 204, 207,
              209, 210, 212, 219, 224, 234, 246, 246, 246, 247, 248, 249, 251,
              252, 264, 267, 270, 271, 271, 273, 274, 275, 278, 282, 283, 286],
             dtype=int32)                                                      ,
       array([  1,   4,   6,   8,   8,  10,  19,  19,  19,  22,  38,  39,  40,
               52,  55,  65,  67,  69,  70,  70,  70,  73,  86,  89,  91,  92,
               96,  97, 103, 107, 115, 121, 125, 126, 126, 129, 130, 130, 133,
              134, 134, 138, 152, 155, 167, 176, 182, 185, 189, 189, 194, 196,
              202, 202, 207, 207, 208, 210, 215, 215, 218, 219, 220, 225, 226,
              227, 231, 238, 241, 244, 251, 259, 

In [117]:
coordinates.shape

(3, 25000, 288)

In [118]:
e_channels, e_z

(array([array([ 19,  23,  43,  48,  55,  60,  62,  67,  70,  70,  73,  77,  85,
                85,  86,  95, 101, 103, 111, 112, 112, 114, 117, 123, 127, 128,
               128, 128, 128, 139, 142, 145, 145, 145, 145, 145, 147, 149, 157,
               160, 160, 160, 163, 175, 176, 179, 202, 202, 203, 203, 204, 207,
               209, 210, 212, 219, 224, 234, 246, 246, 246, 247, 248, 249, 251,
               252, 264, 267, 270, 271, 271, 273, 274, 275, 278, 282, 283, 286],
              dtype=int32)                                                      ,
        array([  1,   4,   6,   8,   8,  10,  19,  19,  19,  22,  38,  39,  40,
                52,  55,  65,  67,  69,  70,  70,  70,  73,  86,  89,  91,  92,
                96,  97, 103, 107, 115, 121, 125, 126, 126, 129, 130, 130, 133,
               134, 134, 138, 152, 155, 167, 176, 182, 185, 189, 189, 194, 196,
               202, 202, 207, 207, 208, 210, 215, 215, 218, 219, 220, 225, 226,
               227, 231, 238, 241, 24